# 🏀 Synthetisches Basketball Tracking - 6m Hoher Korb

## 🎯 Problemstellung

Ein revolutionäres Basketball-System mit einem **6 Meter hohen Korb**! Wir generieren synthetische Daten und verwenden den Kalman Filter um:

- **Synthetische Basketball-Trajektorien zu generieren**, die garantiert im 6m hohen Korb landen
- **Realistische Physik-Simulation** mit angepasster Wurfmechanik für die extreme Höhe
- **Live-Vorhersage der Landung** mit Kalman Filter Tracking
- **Performance-Analyse** der Vorhersagegenauigkeit

## 🏗️ Technische Spezifikationen

### 🏀 Basketball Court (Modifiziert)
- **Korb-Höhe**: 6.0 m (doppelt so hoch wie normal!)
- **Court-Länge**: 28.65 m (NBA Standard)
- **Korb-Durchmesser**: 0.45 m
- **Schwerkraft**: 9.81 m/s²

### 📊 Kalman Filter Setup
- **Zustandsvektor**: $\mathbf{x} = [x, y, z, v_x, v_y, v_z]^T$
- **Messungen**: Kamera-Position $(x, y, z)$ mit realistischem Rauschen
- **Physik-Modell**: Eingebaute Schwerkraft und Luftwiderstand

### 🎮 Synthetische Daten
- **Garantierte Treffer**: Alle generierten Würfe landen im 6m Korb
- **Verschiedene Startpositionen**: Freiwürfe, 3-Punkte-Würfe, etc.
- **Realistische Rauschmodelle**: Kamera-Messungenauigkeiten

In [1]:
# 📚 Import aller benötigten Bibliotheken
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.animation as animation
from IPython.display import HTML, clear_output
import time
import warnings
import scipy.optimize as opt
import pandas as pd
warnings.filterwarnings('ignore')

# Matplotlib Setup für interaktive Plots
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.style.use('seaborn-v0_8')

# Live-Plotting aktivieren
import matplotlib
matplotlib.use('Qt5Agg')
%matplotlib qt
plt.ion()

# Reproduzierbare Ergebnisse
np.random.seed(42)

print("✅ Alle Bibliotheken erfolgreich importiert!")
print("🏀 System bereit für 6m Basketball-Tracking!")
print("📊 Synthetische Daten-Generation aktiviert!")
print("📺 Live-Plotting funktionsbereit!")

✅ Alle Bibliotheken erfolgreich importiert!
🏀 System bereit für 6m Basketball-Tracking!
📊 Synthetische Daten-Generation aktiviert!
📺 Live-Plotting funktionsbereit!


In [2]:
class SimpleBasketballGenerator:
    """
    Einfacher Generator für synthetische Basketball-Daten
    Korb-Höhe: 3m, verschiedene Abwurf-Distanzen
    """
    
    def __init__(self):
        # Basketball-Parameter
        self.g = 9.81  # Schwerkraft [m/s²]
        self.basket_height = 3.0  # [m] - Standard Korb
        self.basket_radius = 0.225  # [m]
        self.throw_height = 2.0  # [m] - Abwurf-Höhe
        
        # Distanzen mit entsprechenden Datenpunkt-Anzahlen (OPTIMIERT)
        self.distances = [2, 4, 6, 8, 10, 20, 30]  # [m]
        self.datapoints_per_distance = {
            2: 5,     # Kurze Distanz - wenige Punkte (reduziert von 20)
            4: 8,     # (reduziert von 25)
            6: 10,    # (reduziert von 30)
            8: 12,    # (reduziert von 35)
            10: 15,   # (reduziert von 40)
            20: 20,   # Mittlere Distanz (reduziert von 60)
            30: 25    # Lange Distanz (reduziert von 80)
        }
        
        print(f"🏀 Einfacher Basketball-Generator initialisiert!")
        print(f"📏 Korb-Höhe: {self.basket_height:.1f} m")
        print(f"🎯 Distanzen: {self.distances} m")
        
    def calculate_perfect_throw(self, distance):
        """
        Berechnet perfekte Wurfparameter für garantierten Treffer
        """
        # Höhenunterschied
        dz = self.basket_height - self.throw_height  # 3.0 - 2.0 = 1.0 m
        
        # Optimaler Wurfwinkel (45° + Korrektur für Höhe)
        angle = np.deg2rad(45 + dz * 2)  # Leicht steiler für höheren Korb
        
        # Anfangsgeschwindigkeit für exakte Landung
        v0 = np.sqrt(distance * self.g / np.sin(2 * angle))
        
        # Geschwindigkeitskomponenten
        vx = v0 * np.cos(angle)
        vz = v0 * np.sin(angle)
        
        # Flugzeit
        flight_time = distance / vx
        
        return v0, angle, vx, vz, flight_time
    
    def generate_trajectory(self, distance, dt=0.05):  # OPTIMIERT: dt größer
        """
        Generiert perfekte Trajektorie für gegebene Distanz (OPTIMIERT)
        """
        v0, angle, vx, vz, flight_time = self.calculate_perfect_throw(distance)
        
        # Simulation (OPTIMIERT: weniger Zeitschritte)
        trajectory_points = []
        t = 0.0
        
        while t <= flight_time:
            x = vx * t
            z = self.throw_height + vz * t - 0.5 * self.g * t**2
            
            # Stoppe wenn Ball den Boden erreicht oder Korb-Position
            if z <= 0 or (abs(x - distance) < 0.1 and abs(z - self.basket_height) < 0.2):
                break
                
            trajectory_points.append([t, x, 0.0, z])  # y=0 (gerader Wurf)
            t += dt
        
        # Stelle sicher, dass letzter Punkt genau im Korb ist
        if len(trajectory_points) > 0:
            trajectory_points[-1] = [t, distance, 0.0, self.basket_height]
        
        return np.array(trajectory_points)
    
    def add_measurement_noise(self, trajectory, noise_std=0.02):
        """
        Fügt realistisches Messrauschen hinzu
        """
        noisy_traj = trajectory.copy()
        
        # Rauschen nur für Position (x,y,z), nicht für Zeit
        noise = np.random.normal(0, noise_std, (len(trajectory), 3))
        noisy_traj[:, 1:4] += noise
        
        return noisy_traj
    
    def generate_all_data(self):
        """
        Generiert alle synthetischen Daten für verschiedene Distanzen (OPTIMIERT)
        """
        print("🎯 Generiere synthetische Basketball-Daten (OPTIMIERT)...")
        
        all_data = []
        shot_id = 0
        
        # Progress-Tracking
        total_shots = sum(self.datapoints_per_distance.values())
        print(f"📊 Insgesamt {total_shots} Würfe zu generieren...")
        
        for distance in self.distances:
            num_points = self.datapoints_per_distance[distance]
            print(f"📊 Distanz {distance}m: {num_points} Datenpunkte")
            
            for shot_num in range(num_points):
                # Kleine Variation für Realismus
                varied_distance = distance + np.random.normal(0, 0.1)
                varied_distance = max(1.0, varied_distance)  # Mindestens 1m
                
                # Generiere Trajektorie
                trajectory = self.generate_trajectory(varied_distance)
                
                # Füge Messrauschen hinzu
                noisy_trajectory = self.add_measurement_noise(trajectory)
                
                # OPTIMIERT: Batch-Processing für Datenpunkte
                shot_data = []
                for point_idx, (true_point, noisy_point) in enumerate(zip(trajectory, noisy_trajectory)):
                    row = {
                        'shot_id': shot_id,
                        'distance_category': distance,
                        'actual_distance': varied_distance,
                        'point_id': point_idx,
                        'time': true_point[0],
                        'true_x': true_point[1],
                        'true_y': true_point[2], 
                        'true_z': true_point[3],
                        'measured_x': noisy_point[1],
                        'measured_y': noisy_point[2],
                        'measured_z': noisy_point[3],
                        'basket_height': self.basket_height,
                        'guaranteed_hit': True
                    }
                    shot_data.append(row)
                
                # OPTIMIERT: Batch-Append statt einzeln
                all_data.extend(shot_data)
                shot_id += 1
                
                # Progress alle 10 Würfe statt 50
                if shot_id % 10 == 0:
                    progress = (shot_id / total_shots) * 100
                    print(f"  ✅ {shot_id}/{total_shots} Würfe ({progress:.1f}%) generiert...")
        
        df = pd.DataFrame(all_data)
        
        print(f"\n🎯 DATEN-GENERATION ABGESCHLOSSEN!")
        print(f"📊 {shot_id} Würfe generiert (reduziert für Performance)")
        print(f"📋 {len(df)} Trajektorien-Punkte")
        print(f"🏀 Alle Würfe landen garantiert im 3m Korb!")
        print(f"⚡ OPTIMIERT: ~5x schneller durch weniger Datenpunkte!")
        
        return df

# Initialisiere Generator
basketball_gen = SimpleBasketballGenerator()

print("🏗️ Einfacher Basketball-Generator bereit!")
print("🎯 Bereit für Daten-Generation!")

🏀 Einfacher Basketball-Generator initialisiert!
📏 Korb-Höhe: 3.0 m
🎯 Distanzen: [2, 4, 6, 8, 10, 20, 30] m
🏗️ Einfacher Basketball-Generator bereit!
🎯 Bereit für Daten-Generation!


In [ ]:
# 🎯 DATEN GENERIERUNG UND CSV EXPORT

print("🏀" + "="*60 + "🏀")
print("    SYNTHETISCHE BASKETBALL-DATEN GENERATION")
print("    KORB-HÖHE: 3m | DISTANZEN: 2-30m")
print("🏀" + "="*60 + "🏀")

# 1. Generiere alle Daten
print("\n📊 Generiere synthetische Daten...")
basketball_data = basketball_gen.generate_all_data()

# 2. Speichere als einzige CSV-Datei
csv_filename = "basketball_synthetic_data.csv"
basketball_data.to_csv(csv_filename, index=False)

print(f"\n💾 CSV-Datei gespeichert: {csv_filename}")
print(f"? Datei-Größe: {len(basketball_data)} Zeilen")

# 3. Zeige Datenübersicht
print(f"\n📋 DATENÜBERSICHT:")
print(f"📊 Spalten: {list(basketball_data.columns)}")
print(f"\n📈 Statistik nach Distanz:")
distance_stats = basketball_data.groupby('distance_category').agg({
    'shot_id': 'nunique',
    'point_id': 'max',
    'time': 'max',
    'actual_distance': ['mean', 'std']
}).round(3)
print(distance_stats)

# 4. Erste Zeilen anzeigen
print(f"\n👀 Erste 10 Zeilen:")
print(basketball_data[['shot_id', 'distance_category', 'time', 'measured_x', 'measured_z']].head(10))

print(f"\n✅ Synthetische Daten erfolgreich generiert und gespeichert!")
print(f"🚀 Bereit für Kalman Filter Analyse!")

In [ ]:
class BasketballKalmanFilter:
    """
    Kalman Filter für Basketball-Trajektorien Vorhersage
    """
    
    def __init__(self, dt=0.02):
        self.dt = dt
        self.g = 9.81
        
        # Zustandsvektor: [x, z, vx, vz] (nur 2D für Einfachheit)
        self.x = np.zeros(4)
        self.P = np.eye(4) * 10.0
        
        # Zustandsübergangsmatrix
        self.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])
        
        # Kontrollmatrix für Schwerkraft
        self.B = np.array([0, 0, 0, -self.g]).reshape(4, 1)
        self.u = np.array([dt])
        
        # Beobachtungsmatrix (messen x, z)
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])
        
        # Rauschmatrizen
        self.Q = np.eye(4) * 0.1  # Prozessrauschen
        self.R = np.eye(2) * 0.02  # Messrauschen
        
        self.initialized = False
        self.history = []
        
    def initialize(self, first_measurement):
        """Initialisiert Filter mit erster Messung"""
        self.x[0] = first_measurement[0]  # x
        self.x[1] = first_measurement[1]  # z
        self.x[2] = 0  # vx (wird geschätzt)
        self.x[3] = 0  # vz (wird geschätzt)
        self.initialized = True
        self.history.append(self.x.copy())
        
    def predict(self):
        """Vorhersage-Schritt"""
        self.x = self.F @ self.x + (self.B @ self.u).flatten()
        self.P = self.F @ self.P @ self.F.T + self.Q
        
    def update(self, measurement):
        """Update-Schritt"""
        z = np.array([measurement[0], measurement[1]])  # x, z
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        
    def step(self, measurement):
        """Kompletter Kalman Filter Schritt"""
        if not self.initialized:
            self.initialize(measurement)
            return
            
        self.predict()
        self.update(measurement)
        self.history.append(self.x.copy())
        
    def predict_landing(self, basket_height=3.0):
        """Vorhersage wo Ball landen wird"""
        if self.x[3] >= 0:  # Ball steigt noch
            return None, None
            
        # Berechne Zeit bis Ball basket_height erreicht
        current_z = self.x[1]
        current_vz = self.x[3]
        
        # Quadratische Gleichung lösen: z = current_z + vz*t - 0.5*g*t²
        a = -0.5 * self.g
        b = current_vz
        c = current_z - basket_height
        
        discriminant = b**2 - 4*a*c
        if discriminant < 0:
            return None, None
            
        t1 = (-b + np.sqrt(discriminant)) / (2*a)
        t2 = (-b - np.sqrt(discriminant)) / (2*a)
        
        # Wähle positive Zeit in der Zukunft
        t_landing = t1 if t1 > 0 else t2
        if t_landing <= 0:
            return None, None
            
        # Berechne x-Position bei Landung
        x_landing = self.x[0] + self.x[2] * t_landing
        
        return x_landing, t_landing

print("? Basketball Kalman Filter implementiert!")
print("? Bereit für Live-Vorhersagen!")

📁 Verzeichnis 'basketball_6m_data' erstellt
💾 CSV-Exporter bereit!
📁 Daten werden gespeichert in: basketball_6m_data/


In [ ]:
# 📊 DATEN LADEN UND KALMAN FILTER DEMO

print("🏀" + "="*60 + "🏀")
print("    LADE SYNTHETISCHE DATEN UND TESTE KALMAN FILTER")
print("🏀" + "="*60 + "🏀")

# 1. Lade die generierten Daten
print("\n📂 Lade Basketball-Daten...")
try:
    df = pd.read_csv("basketball_synthetic_data.csv")
    print(f"✅ Daten geladen: {len(df)} Zeilen")
except FileNotFoundError:
    print("❌ CSV-Datei nicht gefunden. Bitte zuerst Daten generieren!")
    df = None

if df is not None:
    # 2. Zeige Datenübersicht
    print(f"\n📊 DATENÜBERSICHT:")
    print(f"🎯 Unique Würfe: {df['shot_id'].nunique()}")
    print(f"📏 Distanz-Kategorien: {sorted(df['distance_category'].unique())}")
    print(f"⏱️  Max. Flugzeit: {df['time'].max():.2f} s")
    
    # 3. Wähle einen Testwurf für Kalman Filter Demo
    print(f"\n🎯 Wähle Testwurf für Kalman Filter Demo...")
    
    # Nimm einen mittleren Wurf (10m Distanz)
    test_shots = df[df['distance_category'] == 10]['shot_id'].unique()
    test_shot_id = test_shots[0]
    
    test_data = df[df['shot_id'] == test_shot_id].sort_values('time')
    
    print(f"📋 Testwurf ausgewählt:")
    print(f"   Shot ID: {test_shot_id}")
    print(f"   Distanz: {test_data['distance_category'].iloc[0]}m")
    print(f"   Datenpunkte: {len(test_data)}")
    print(f"   Flugzeit: {test_data['time'].max():.2f}s")
    
    # 4. Bereite Kalman Filter vor
    kf = BasketballKalmanFilter(dt=0.02)
    
    # Extrahiere Messungen (x, z)
    measurements = test_data[['measured_x', 'measured_z']].values
    true_positions = test_data[['true_x', 'true_z']].values
    times = test_data['time'].values
    
    print(f"\n🔬 Kalman Filter Test startet...")
    print(f"📊 {len(measurements)} Messungen werden verarbeitet")
    
    # 5. Führe Kalman Filter durch
    predictions = []
    landing_predictions = []
    
    for i, measurement in enumerate(measurements):
        kf.step(measurement)
        
        # Vorhersage der Landungsposition
        x_landing, t_landing = kf.predict_landing()
        
        predictions.append({
            'time': times[i],
            'measurement_x': measurement[0],
            'measurement_z': measurement[1],
            'true_x': true_positions[i][0],
            'true_z': true_positions[i][1],
            'kalman_x': kf.x[0],
            'kalman_z': kf.x[1],
            'kalman_vx': kf.x[2],
            'kalman_vz': kf.x[3],
            'predicted_landing_x': x_landing if x_landing is not None else np.nan,
            'predicted_landing_time': t_landing if t_landing is not None else np.nan
        })
    
    predictions_df = pd.DataFrame(predictions)
    
    print(f"✅ Kalman Filter Test abgeschlossen!")
    print(f"📊 Vorhersagen erstellt für {len(predictions)} Zeitpunkte")
    
    # 6. Zeige Ergebnisse
    print(f"\n🎯 KALMAN FILTER ERGEBNISSE:")
    
    # Finale Landungsvorhersage
    final_prediction = predictions_df['predicted_landing_x'].dropna()
    if len(final_prediction) > 0:
        final_landing_pred = final_prediction.iloc[-1]
        true_landing = true_positions[-1][0]  # Wahre finale x-Position
        
        print(f"🏀 Finale Landungsvorhersage:")
        print(f"   Vorhergesagt: {final_landing_pred:.2f} m")
        print(f"   Tatsächlich:  {true_landing:.2f} m")
        print(f"   Fehler:       {abs(final_landing_pred - true_landing):.3f} m")
        
        # Treffer-Analyse (Korb hat 0.45m Durchmesser)
        basket_center = test_data['distance_category'].iloc[0]  # Target x-position
        distance_to_basket = abs(final_landing_pred - basket_center)
        hit = distance_to_basket <= 0.225  # Korb-Radius
        
        print(f"🎯 Treffer-Analyse:")
        print(f"   Abstand zum Korb: {distance_to_basket:.3f} m")
        print(f"   Treffer: {'✅ JA' if hit else '❌ NEIN'}")
    
    print(f"\n📊 Erste Kalman Filter Vorhersagen:")
    print(predictions_df[['time', 'kalman_x', 'kalman_z', 'predicted_landing_x']].head(10).round(3))
    
    print(f"\n🚀 Bereit für Live-Visualisierung!")
    
    # Mache Daten global verfügbar für nächste Zelle
    globals()['test_data'] = test_data
    globals()['predictions_df'] = predictions_df
    globals()['basketball_data'] = df

🏀======================================================================🏀
         SYNTHETISCHE BASKETBALL-DATEN GENERATION
         6 METER HOHER KORB - GARANTIERTE TREFFER
🏀======================================================================🏀

🎯 Schritt 1: Generiere synthetische Basketball-Würfe...
🏀============================================================🏀
    SYNTHETISCHE BASKETBALL-DATEN GENERATION
🏀============================================================🏀
🎯 Generiere Datensatz: freethrow_6m
📝 Freiwürfe zum 6m Korb
  ✅ 20 Würfe generiert...
  ✅ 40 Würfe generiert...
  ✅ 60 Würfe generiert...
  ✅ 80 Würfe generiert...
✅ Datensatz 'freethrow_6m' komplett: 90 Würfe
🎯 Generiere Datensatz: three_point_6m
📝 3-Punkt-Würfe zum 6m Korb
  ✅ 20 Würfe generiert...
  ✅ 40 Würfe generiert...
  ✅ 60 Würfe generiert...
  ✅ 80 Würfe generiert...
  ✅ 100 Würfe generiert...
  ✅ 120 Würfe generiert...
  ✅ 140 Würfe generiert...
✅ Datensatz 'three_point_6m' komplett: 150 Würfe
🎯 Generiere Date

In [ ]:
# ? LIVE KALMAN FILTER VISUALISIERUNG

class LiveBasketballTracker:
    """
    Live-Visualisierung des Kalman Filter Basketball-Trackings
    Dasselbe Fenster wird kontinuierlich aktualisiert
    """
    
    def __init__(self):
        self.fig = None
        self.axes = None
        self.setup_live_window()
        
    def setup_live_window(self):
        """Erstellt persistentes Visualisierungs-Fenster"""
        plt.ioff()  # Deaktiviere interaktiven Modus temporär
        
        self.fig, self.axes = plt.subplots(2, 2, figsize=(16, 12))
        self.fig.suptitle('🏀 Live Basketball Kalman Filter Tracking', fontsize=16, fontweight='bold')
        
        # Setup der Subplots
        self.axes[0,0].set_title('Trajektorie (2D)')
        self.axes[0,0].set_xlabel('X Position [m]')
        self.axes[0,0].set_ylabel('Z Höhe [m]')
        self.axes[0,0].grid(True, alpha=0.3)
        
        self.axes[0,1].set_title('Kalman Filter Schätzung vs. Wahrheit')
        self.axes[0,1].set_xlabel('Zeit [s]')
        self.axes[0,1].set_ylabel('X Position [m]')
        self.axes[0,1].grid(True, alpha=0.3)
        
        self.axes[1,0].set_title('Landungsvorhersage über Zeit')
        self.axes[1,0].set_xlabel('Zeit [s]')
        self.axes[1,0].set_ylabel('Vorhergesagte X-Landung [m]')
        self.axes[1,0].grid(True, alpha=0.3)
        
        self.axes[1,1].set_title('Live Info Panel')
        self.axes[1,1].axis('off')
        
        plt.tight_layout()
        plt.ion()  # Reaktiviere interaktiven Modus
        plt.show()
        
    def update_visualization(self, current_step, test_data, predictions_df, basket_distance):
        """Aktualisiert die Live-Visualisierung"""
        
        # Lösche vorherige Plots
        for ax in self.axes.flat:
            if ax != self.axes[1,1]:  # Info Panel nicht löschen
                ax.clear()
        
        # Aktuelle Daten bis zum aktuellen Zeitschritt
        current_predictions = predictions_df.iloc[:current_step+1]
        current_test = test_data.iloc[:current_step+1]
        
        # 1. Trajektorie Plot
        ax1 = self.axes[0,0]
        
        # Wahre Trajektorie (komplett, grau)
        ax1.plot(test_data['true_x'], test_data['true_z'], 'gray', alpha=0.3, linewidth=1, label='Wahre Trajektorie (komplett)')
        
        # Kalman Filter Schätzung (bisher)
        if len(current_predictions) > 0:
            ax1.plot(current_predictions['kalman_x'], current_predictions['kalman_z'], 'b-', linewidth=3, label='Kalman Filter')
            
        # Messungen (bisher)
        ax1.scatter(current_test['measured_x'], current_test['measured_z'], color='red', s=30, alpha=0.7, label='Messungen', zorder=5)
        
        # Korb Position
        ax1.axhline(y=3.0, color='orange', linestyle='--', alpha=0.7, label='Korb-Höhe (3m)')
        ax1.axvline(x=basket_distance, color='orange', linestyle='--', alpha=0.7, label=f'Korb-Position ({basket_distance}m)')
        
        ax1.set_title('Trajektorie (2D)')
        ax1.set_xlabel('X Position [m]')
        ax1.set_ylabel('Z Höhe [m]')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(-1, basket_distance + 2)
        ax1.set_ylim(0, 8)
        
        # 2. X-Position über Zeit
        ax2 = self.axes[0,1]
        
        if len(current_predictions) > 0:
            ax2.plot(current_predictions['time'], current_predictions['true_x'], 'g-', linewidth=2, label='Wahre Position')
            ax2.plot(current_predictions['time'], current_predictions['kalman_x'], 'b-', linewidth=2, label='Kalman Filter')
            ax2.scatter(current_predictions['time'], current_predictions['measurement_x'], color='red', s=20, alpha=0.7, label='Messungen')
        
        ax2.axhline(y=basket_distance, color='orange', linestyle='--', alpha=0.7, label=f'Korb ({basket_distance}m)')
        ax2.set_title('Kalman Filter Schätzung vs. Wahrheit')
        ax2.set_xlabel('Zeit [s]')
        ax2.set_ylabel('X Position [m]')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Landungsvorhersage über Zeit
        ax3 = self.axes[1,0]
        
        valid_predictions = current_predictions['predicted_landing_x'].dropna()
        if len(valid_predictions) > 0:
            valid_times = current_predictions.loc[valid_predictions.index, 'time']
            ax3.plot(valid_times, valid_predictions, 'purple', linewidth=3, label='Landungsvorhersage')
            ax3.scatter(valid_times.iloc[-1], valid_predictions.iloc[-1], color='purple', s=100, zorder=5)
        
        ax3.axhline(y=basket_distance, color='orange', linestyle='--', alpha=0.7, label=f'Korb-Ziel ({basket_distance}m)')
        ax3.set_title('Landungsvorhersage über Zeit')
        ax3.set_xlabel('Zeit [s]')
        ax3.set_ylabel('Vorhergesagte X-Landung [m]')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Info Panel
        ax4 = self.axes[1,1]
        ax4.clear()
        ax4.axis('off')
        
        if current_step < len(predictions_df):
            current_pred = predictions_df.iloc[current_step]
            current_test_point = test_data.iloc[current_step]
            
            # Landungsvorhersage
            landing_pred = current_pred['predicted_landing_x']
            landing_text = f"{landing_pred:.2f} m" if not pd.isna(landing_pred) else "Noch nicht verfügbar"
            
            # Abstand zum Korb
            if not pd.isna(landing_pred):
                distance_to_basket = abs(landing_pred - basket_distance)
                hit_prediction = "✅ TREFFER" if distance_to_basket <= 0.225 else "❌ VERFEHLT"
            else:
                distance_to_basket = np.nan
                hit_prediction = "⏳ Warte auf Vorhersage..."
            
            info_text = f"""
🏀 LIVE BASKETBALL TRACKING

📊 Aktueller Status:
   Zeit: {current_pred['time']:.3f} s
   Schritt: {current_step + 1} / {len(predictions_df)}

📍 Kalman Filter Zustand:
   Position X: {current_pred['kalman_x']:.2f} m
   Position Z: {current_pred['kalman_z']:.2f} m
   Geschw. X:  {current_pred['kalman_vx']:.2f} m/s
   Geschw. Z:  {current_pred['kalman_vz']:.2f} m/s

🎯 Landungsvorhersage:
   Vorhergesagt: {landing_text}
   Korb-Position: {basket_distance:.1f} m
   Abstand: {distance_to_basket:.3f} m
   
🏀 Vorhersage: {hit_prediction}

📏 Messung vs. Filter:
   Mess-X: {current_pred['measurement_x']:.2f} m
   Filter-X: {current_pred['kalman_x']:.2f} m
   Differenz: {abs(current_pred['measurement_x'] - current_pred['kalman_x']):.3f} m
            """
            
            ax4.text(0.05, 0.95, info_text, transform=ax4.transAxes, 
                    fontsize=11, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        # Update der Fenster-Titel
        self.fig.suptitle(f'🏀 Live Basketball Kalman Filter - Schritt {current_step+1}/{len(predictions_df)}', 
                         fontsize=16, fontweight='bold')
        
        # Force update
        plt.draw()
        plt.pause(0.01)

def run_live_demo():
    """Führt Live-Demo mit den geladenen Daten durch"""
    
    if 'test_data' not in globals() or 'predictions_df' not in globals():
        print("❌ Keine Testdaten verfügbar! Bitte zuerst vorherige Zelle ausführen.")
        return
    
    print("🎬 Starte Live Basketball Tracking Demo...")
    
    # Erstelle Live Tracker
    tracker = LiveBasketballTracker()
    
    # Hole Testdaten
    test_data = globals()['test_data']
    predictions_df = globals()['predictions_df']
    basket_distance = test_data['distance_category'].iloc[0]
    
    print(f"📊 Demo läuft mit {len(predictions_df)} Schritten...")
    print(f"🎯 Korb-Distanz: {basket_distance} m")
    print(f"⏸️  Verwenden Sie Strg+C um zu stoppen")
    
    try:
        # Live-Animation
        for step in range(len(predictions_df)):
            tracker.update_visualization(step, test_data, predictions_df, basket_distance)
            time.sleep(0.5)  # Pause zwischen Updates
            
            # Status-Update alle 10 Schritte
            if (step + 1) % 10 == 0:
                print(f"📊 Schritt {step + 1}/{len(predictions_df)} abgeschlossen")
    
    except KeyboardInterrupt:
        print("\n⏸️  Demo gestoppt durch Benutzer")
    
    print(f"\n✅ Live Demo abgeschlossen!")
    print(f"? Fenster bleibt für weitere Analysen geöffnet")

# Starte Live Demo
if 'test_data' in globals():
    run_live_demo()
else:
    print("⚠️  Bitte zuerst Daten laden (vorherige Zelle ausführen)")